# Lesson 6.1 — Grafana AppDynamics Datasource: Setup and Configuration

## Learning Objectives
- Install the `dlopes7-appdynamics-datasource` Grafana plugin
- Configure the datasource with Controller URL, credentials, and Analytics API key
- Test the datasource connection
- Understand the difference between Controller API auth and Analytics API auth

**Reference**: [Grafana AppDynamics Plugin](https://grafana.com/docs/plugins/dlopes7-appdynamics-datasource/latest/) | [Configure Datasource](https://grafana.com/docs/plugins/dlopes7-appdynamics-datasource/latest/configure/)

## 1. Why Use Grafana with AppDynamics?

While AppDynamics has its own dashboard system (Dash Studio), there are reasons to use Grafana:

| Reason | Details |
|--------|--------|
| **Multi-source dashboards** | Combine AppDynamics metrics with Prometheus, Loki, Elasticsearch on one dashboard |
| **Familiar interface** | Teams already using Grafana can stay in one tool |
| **Advanced visualization** | Grafana has richer visualization options (heatmaps, histograms, etc.) |
| **Alerting** | Grafana's unified alerting can notify alongside AppDynamics alerts |
| **Annotations** | AppDynamics Events can appear as annotations on any Grafana panel |

### The Plugin
The community plugin `dlopes7-appdynamics-datasource` is published on Grafana.com and actively maintained.

**Plugin page**: https://grafana.com/grafana/plugins/dlopes7-appdynamics-datasource/

## 2. Installing the Plugin

### Method 1: Grafana CLI
```bash
grafana-cli plugins install dlopes7-appdynamics-datasource
```

### Method 2: Grafana UI (v10+)
1. Go to **Connections → Add new connection**
2. Search for "AppDynamics"
3. Click **Install** on the `dlopes7-appdynamics-datasource` plugin

### Method 3: Docker / Kubernetes
```bash
# Docker
docker run -d \
  -e GF_INSTALL_PLUGINS=dlopes7-appdynamics-datasource \
  -p 3000:3000 \
  grafana/grafana

# Helm chart (grafana/grafana)
# values.yaml:
# plugins:
#   - dlopes7-appdynamics-datasource
```

### After installation:
- Restart Grafana: `systemctl restart grafana-server`
- The plugin will appear in **Connections → Data sources → Add data source**

In [ ]:
# Verify Grafana and plugin installation

import subprocess, json

# Check if grafana-cli is available
try:
    result = subprocess.run(['grafana-cli', '--version'], capture_output=True, text=True, timeout=5)
    print(f"Grafana CLI: {result.stdout.strip() or result.stderr.strip()}")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("grafana-cli not found in PATH.")
    print("Grafana may be running as a service, or you may need to add it to PATH.")
    print()
    print("Install commands:")
    print("  macOS (Homebrew): brew install grafana")
    print("  Ubuntu/Debian   : sudo apt-get install grafana")
    print("  Windows         : winget install GrafanaLabs.Grafana")
    print("  Docker          : docker run -p 3000:3000 grafana/grafana")

# List installed plugins (if grafana-cli is available)
try:
    result = subprocess.run(['grafana-cli', 'plugins', 'ls'], capture_output=True, text=True, timeout=10)
    if 'appdynamics' in result.stdout.lower():
        print("✅ AppDynamics plugin is installed")
    else:
        print("AppDynamics plugin not found. Install with:")
        print("  grafana-cli plugins install dlopes7-appdynamics-datasource")
except:
    pass

## 3. Configuring the Datasource

### Steps in Grafana UI:
1. Navigate to **Connections → Data sources → Add data source**
2. Search for and select **AppDynamics**
3. Fill in the configuration form:

### Required fields:

| Field | Description | Example |
|-------|-------------|----------|
| **Name** | Datasource label in Grafana | `AppDynamics Production` |
| **Controller URL** | Your Controller Tenant URL | `https://acme.saas.appdynamics.com` |
| **Username** | AppDynamics username | `admin` |
| **Password** | AppDynamics password | (secure input) |
| **Account Name** | Your AppDynamics account name | `acme-corp` |

### Optional (for Analytics/ADQL queries):

| Field | Description |
|-------|-------------|
| **Analytics API URL** | Analytics Events API endpoint | `https://analytics.api.appdynamics.com` |
| **Analytics API Key** | API key for Analytics authentication |
| **Global Account Name** | Found in Account Management Portal |

> **Important**: Analytics queries use DIFFERENT credentials than Metrics queries. The Analytics API Key is generated in the AppDynamics Account Management Portal, not the Controller UI.

**Reference**: [Datasource Configuration](https://grafana.com/docs/plugins/dlopes7-appdynamics-datasource/latest/configure/)

In [ ]:
# Verify datasource connectivity using Grafana's API
# Once the datasource is configured, Grafana exposes it via REST API

import os, requests

GRAFANA_URL  = os.getenv("GRAFANA_URL",  "http://localhost:3000")
GRAFANA_USER = os.getenv("GRAFANA_USER", "admin")
GRAFANA_PASS = os.getenv("GRAFANA_PASS", "admin")

def list_grafana_datasources():
    """List all configured datasources in Grafana."""
    url  = f"{GRAFANA_URL}/api/datasources"
    auth = (GRAFANA_USER, GRAFANA_PASS)
    resp = requests.get(url, auth=auth, timeout=10)
    resp.raise_for_status()
    return resp.json()

try:
    datasources = list_grafana_datasources()
    print(f"Grafana datasources ({len(datasources)} found):")
    for ds in datasources:
        mark = "✅" if 'appdynamics' in ds['type'].lower() else "  "
        print(f"  {mark} [{ds['type']:<30}] {ds['name']}")
    
    appd_ds = [ds for ds in datasources if 'appdynamics' in ds['type'].lower()]
    if appd_ds:
        print(f"\n✅ AppDynamics datasource found: '{appd_ds[0]['name']}'")
    else:
        print("\n❌ No AppDynamics datasource configured yet.")

except Exception as e:
    print(f"Could not connect to Grafana at {GRAFANA_URL}")
    print(f"Error: {e}")
    print()
    print("Set environment variables:")
    print("  GRAFANA_URL  = http://your-grafana-host:3000")
    print("  GRAFANA_USER = admin")
    print("  GRAFANA_PASS = your-password")

In [ ]:
# EXERCISE 6.1: Datasource configuration checklist

config_checklist = [
    {
        "item": "Grafana is running and accessible",
        "check": "Can you open http://localhost:3000 in a browser?",
        "status": "TODO"  # Change to DONE when complete
    },
    {
        "item": "Plugin is installed",
        "check": "Run: grafana-cli plugins ls | grep appdynamics",
        "status": "TODO"
    },
    {
        "item": "Controller URL is reachable",
        "check": "Open https://your-account.saas.appdynamics.com in browser",
        "status": "TODO"
    },
    {
        "item": "Datasource configured with Controller credentials",
        "check": "Grafana UI: Connections > Data sources > AppDynamics > Save & Test shows OK",
        "status": "TODO"
    },
    {
        "item": "Analytics API key obtained (for ADQL)",
        "check": "Account Management Portal > Analytics > API Keys > Generate",
        "status": "TODO"
    },
    {
        "item": "Analytics URL and key configured in datasource",
        "check": "Grafana datasource: Analytics API URL and Analytics API Key fields filled in",
        "status": "TODO"
    },
]

done = sum(1 for c in config_checklist if c['status'] == 'DONE')
print(f"Setup Progress: {done}/{len(config_checklist)} items complete")
print()
for item in config_checklist:
    icon = "✅" if item['status'] == 'DONE' else "❌"
    print(f"  {icon} {item['item']}")
    if item['status'] != 'DONE':
        print(f"      → {item['check']}")

## Summary

- Install the plugin via `grafana-cli plugins install dlopes7-appdynamics-datasource` or the Grafana UI
- Configure two sets of credentials: **Controller** (username/password/account) and **Analytics** (API key/global account name)
- The **Analytics API credentials** are separate and obtained from the Account Management Portal
- Click **Save & Test** in the datasource configuration to verify connectivity

---

**Next**: [Lesson 6.2 — Metrics Query Type](lesson-6.2-metrics-query-type.ipynb)